In [1]:

import sys, os
sys.path.append(os.path.abspath("..")) 
import json
import torch
from datasets import GANDataset
from models import Generator
from utils import args_gan, gen_image, plotly_generate

**Read configuration files and arguments:**

In [2]:
# Arguments
parser = args_gan()
args, unknown = parser.parse_known_args()

args.particle = "proton_contained"
args.metadata_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/metadata.pkl"
args.dataset_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/{}/{}/{}/{}.zip"
args.gan_ind_path = "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/gan_ind.pkl"
args.save_dir = "/pscratch/sd/b/botaoli/SFGD_VA/Results/gan/"
args.checkpoint_path = "/pscratch/sd/b/botaoli/SFGD_VA/Results/gan/checkpoints"
args.checkpoint_name = args.particle

if args.particle == "muon" or args.particle == "proton_exiting":
    args.label_size = 10
elif args.particle == "proton_contained":
    args.label_size = 7
args.epochs = 50
args.log_every_n_steps = 2000
args.batch_size = 32
args.hidden = 64
args.warmup_steps = 10
args.num_workers = 64

**Load the pre-trained weights of the different generative-adversarial-network (GAN) models:**

In [3]:
print(checkpoint_p['state_dict'].keys())

NameError: name 'checkpoint_p' is not defined

In [3]:
# Dataset and generator models
test_set_p = GANDataset(args, split="test")

# Geneator and critic models
generator = Generator(input_size=args.input_size, label_size=args.label_size, noise_size=args.noise_size,
                          hidden=args.hidden, n_layers=args.layers, attn_heads=args.attn_heads, dropout=args.dropout)

checkpoint_path = "/".join((args.checkpoint_path, args.checkpoint_name, "w_loss", "epoch=2-step=2220000.ckpt"))
# Load weights of pre-trained generator models
checkpoint_p = torch.load(checkpoint_path, map_location='cpu')

state_dict = {
    key.replace("generator.", ""): value for key, value in checkpoint_p['state_dict'].items()
}

generator.load_state_dict(state_dict, strict=False)
generator.eval();

**Run each GAN on some arbitrary input kinematics:**

In [4]:
'''
Proton GAN
'''
import numpy as np

# Set your kinematics here:
# ke = 30.3  # Initial kinetic energy
# ini_dir = [0.9999999999999999, 0.0, 0.0]  # Initial direction
# ini_pos = [-1.5, -4.2, 2.7]  # Initial 3D position (mm)

# get one event from the test set
event = test_set_p[15]
# convert torch tensor to numpy array
ke = float(event['ke'].numpy())
ini_pos = event['pos_ini'].numpy()
ini_dir = event['dir_ini'].numpy()
if args.particle == "proton_exiting" or args.particle == "muon":
    exit_pos = event['pos_exit'].numpy()
else:
    exit_pos = None
img = event['image'].numpy()

print(ke, ini_pos, ini_dir, exit_pos, img)

if exit_pos is not None:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2], exit_pos[0], exit_pos[1], exit_pos[2]])
else:
    params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2]])

# Run the generator!
generated_p = gen_image(generator=generator, args=args, test_set=test_set_p, 
                        ke=ke, ini_dir=ini_dir, ini_pos=ini_pos, exit_pos=exit_pos)[0]



-1.441534066798533 [ 0.17021623  0.17007254 -0.04959614] [ 0.53416944 -0.1691078  -0.82829076] None [-1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -0.78798302 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989 -1.00225989
 -1.00225989 -1.00225989

/tmp/ipykernel_604894/1032562225.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ke = float(event['ke'].numpy())


**Visualise the GAN-generated images:**

In [5]:
'''
Plot the generated images!
'''


#generated_p = generated_p.numpy()
# copy the image to a pure numpy array

print(generated_p.shape)

generated_plot = np.zeros((5, 5, 5))

for i in range(generated_plot.shape[0]):
    for j in range(generated_plot.shape[1]):
        for k in range(generated_plot.shape[2]):
            generated_plot[i, j, k] = float(generated_p[i, j, k])
print(generated_plot)
# check the type of the elements in the array
print(generated_plot.dtype)

# Max deposited energy in one voxel
max_energy = generated_plot.max()

#generated_plot[generated_plot > 150] = 0
#generated_plot[generated_plot < 5] = 0
print(max_energy)




torch.Size([5, 5, 5])
[[[  3.           3.           3.           3.           3.        ]
  [  3.           3.           3.           3.           3.        ]
  [  3.           3.           3.           3.           3.        ]
  [  3.           3.           3.           3.           3.        ]
  [  3.           3.           3.           3.           3.        ]]

 [[  3.           3.           3.           3.           3.        ]
  [  3.           3.           3.           3.           3.        ]
  [  3.           3.           3.00079131   3.           3.        ]
  [  3.           3.           3.           3.           3.        ]
  [  3.           3.           3.           3.           3.        ]]

 [[  3.           3.           3.           3.           3.        ]
  [  3.           3.           3.00142431   3.           3.        ]
  [  3.00015831   6.05011702 531.54241943   3.16790366   3.        ]
  [  3.           8.93620682   3.70381832   3.           3.        ]
  [  3. 

In [14]:
generated_plot = generated_plot.astype(np.float32)

In [6]:

print("- Proton image:")
plotly_generate(generated_plot, max_energy=max_energy)


- Proton image:


In [ ]:
event = test_set_p[15]
true_img = np.zeros((125,))

for i in range(true_img.shape[0]):
    true_img[i] = float(event["image"][i])

true_img = true_img.reshape(5, 5, 5)

# get back the normalization
min_charge = test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['min']
max_charge = test_set_p.metadata['statistics']['per_tree'][args.particle]['recon_charge']['max']
true_img = (true_img + 1) / 2
true_img *= (max_charge - min_charge)
true_img += min_charge

print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['std'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['mean'])

max_energy = true_img.max()

print(true_img)

print("- True image:")
plotly_generate(true_img, max_energy=max_energy)



303.3067761656923
197.1132585084357
[[[1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]
  [1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]
  [1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]
  [1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]
  [1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]]

 [[1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]
  [1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]
  [1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]
  [1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]
  [1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]]

 [[1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261e-13
   1.45661261e-13]
  [1.45661261e-13 1.45661261e-13 1.45661261e-13 1.45661261

In [16]:
print(event["image"].numpy())

[ 197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  266.32901016  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  261.34530654  197.11325851  197.11325851
  197.11325851  263.76361099 1348.25107589  298.85057845  197.11325851
  197.11325851  197.11325851  286.39696517  197.11325851  197.11325851
  197.